In [39]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq

load_dotenv()
groq_api_key = os.getenv('GROQ_API_KEY')

llm = ChatGroq(api_key = groq_api_key, temperature=0, max_tokens=4096, model='llama-3.3-70b-versatile')

In [63]:
path = "../gmhs.txt"
text = ''

with open(path, 'r') as f:
    text = f.read()

In [64]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

docs = [Document(page_content=text)]
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000, 
    chunk_overlap=200
)
split_docs = text_splitter.split_documents(docs)

In [65]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

allowed_nodes = ["Person", "House", "Location", "Event", "Series"]
allowed_relationships = ["PARENT_OF", "MEMBER_OF", "KILLED", "LOCATED_IN", "ALLIED_WITH", "SON_OF", "DAUGHTER_OF"]

llm_transformer = LLMGraphTransformer(llm=llm, allowed_nodes=allowed_nodes)
graph_documents = await llm_transformer.aconvert_to_graph_documents(split_docs)
print(graph_documents)

[GraphDocument(nodes=[Node(id='Westeros', type='Location', properties={}), Node(id='House Targaryen', type='House', properties={}), Node(id='King Viserys I Targaryen', type='Person', properties={}), Node(id='Rhaenyra Targaryen', type='Person', properties={}), Node(id='Aegon Ii Targaryen', type='Person', properties={}), Node(id='Queen Alicent Hightower', type='Person', properties={}), Node(id='House Hightower', type='House', properties={}), Node(id='Daemon Targaryen', type='Person', properties={}), Node(id='Game Of Thrones', type='Series', properties={}), Node(id='House Of The Dragon', type='Series', properties={}), Node(id='Robert Baratheon', type='Person', properties={}), Node(id='Aerys Ii Targaryen', type='Person', properties={}), Node(id='Viserys Targaryen', type='Person', properties={}), Node(id='Daenerys Targaryen', type='Person', properties={}), Node(id='Essos', type='Location', properties={}), Node(id='House Stark', type='House', properties={}), Node(id='Eddard Stark', type='Per

In [66]:
from langchain_community.graphs import Neo4jGraph

In [ ]:
graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"), 
    username=os.getenv("NEO4J_USERNAME"), 
    password=os.getenv("NEO4J_PASSWORD")
)

try:
    graph.add_graph_documents(graph_documents)
    print("Successfully uploaded data to Neo4j.")
except Exception as e:
    print(f"Upload failed: {e}")

graph.refresh_schema()

✅ Successfully uploaded data to Neo4j.


In [68]:
from langchain.chains import GraphCypherQAChain
from langchain_core.prompts.prompt import PromptTemplate

CYPHER_GENERATION_TEMPLATE = """
You are a Neo4j expert. Generate a Cypher query to answer the question.
Use ONLY the provided schema.

Schema:
{schema}

Rules:
1. To find a father/parent, the arrow MUST point TO the child: (p:Person)-[:PARENT_OF]->(c:Person {{name: 'Jon Snow'}})
2. Do not use *0..1; only use direct relationships.
3. Return only the name of the parent.

Question: {question}
Cypher:"""

cypher_prompt = PromptTemplate(
    input_variables=["schema", "question"], 
    template=CYPHER_GENERATION_TEMPLATE
)

chain = GraphCypherQAChain.from_llm(
    llm=llm, 
    graph=graph, 
    verbose=True,
    allow_dangerous_requests=True,
    validate_cypher=True,
    cypher_prompt=cypher_prompt
)

response = chain.invoke({
    "query": "Who is the father of Jon Snow?"
})
print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Person)-[:PARENT_OF]->(c:Person {name: 'Jon Snow'})
RETURN p.name

Full Context:
[{'p.name': 'Rhaegar Targaryen'}, {'p.name': 'Lyanna Stark'}]

> Finished chain.
Rhaegar Targaryen is the father of Jon Snow.
